# OMNet-V3: Magnification-Aware Multi-Task Fusion Framework for Breast Cancer Histopathology

**OMNet-V3** is a dual-branch deep learning architecture combining **EfficientNet-B0** (local morphological features) and **ViT-Tiny/16** (global context representation) with a **Magnification-Aware Adaptive Fusion (MAF)** gating mechanism and a **Hierarchical Multi-Task Loss** on the **BreakHis** dataset (all magnifications: 40X, 100X, 200X, 400X).

---

## Key Innovations

1. **Dual-Branch Pretrained Backbones:**
   - EfficientNet-B0 (1280-d local features) + ViT-Tiny/16 (192-d self-attention global context).
2. **Magnification-Aware Adaptive Fusion (MAF):**
   - Learns a scale-conditioned gating parameter $\alpha \in (0, 1)$ conditioned on the magnification embedding ($e_m \in \mathbb{R}^{64}$), dynamically balancing local CNN details and global ViT structure across 40X to 400X magnifications.
3. **Hierarchical Multi-Task Loss:**
   - Combines primary binary classification (benign vs. malignant, $L_{\text{binary}}$), 8-class histological subtype classification ($L_{\text{subtype}}$), and hierarchical probability consistency ($L_{\text{consistency}}$):
   $$L_{\text{total}} = 0.3 L_{\text{binary}} + 0.6 L_{\text{subtype}} + 0.1 L_{\text{consistency}}$$
4. **Patient-Disjoint 5-Fold Stratified Group Cross-Validation:**
   - Strict `StratifiedGroupKFold` on `patient_id` guaranteeing zero patient identity leakage across folds across all magnifications.
5. **Direct Fast SSD Cache Fetching & Drive Checkpointing:**
   - Direct download via `kagglehub` on local high-speed SSD cache and persistence of all checkpoints, metrics, and plots to `/content/drive/MyDrive/output_v3/`.

# Section 1: Environment Setup

In [3]:
# ============================================================
# Section 1: Environment Setup
# ============================================================

import os
import sys
import json
import random
import math
import warnings
import gc
import zipfile
import shutil
import glob

from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
    Subset,
)

import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.models as tvm

import timm
import kagglehub

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_curve,
    precision_recall_curve,
)

from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
from pytorch_grad_cam import GradCAM, ScoreCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # cuDNN settings apply only to NVIDIA CUDA builds.
    if torch.version.hip is None:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(42)


# ============================================================
# Environment Verification
# ============================================================

print("=" * 60)
print("  OMNet-V3 Environment Setup Complete")
print("=" * 60)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  PyTorch     : {torch.__version__}")
print(f"  HIP / ROCm  : {torch.version.hip}")
print(f"  timm        : {timm.__version__}")
print(f"  GPU Avail   : {torch.cuda.is_available()}")
print("=" * 60)

  OMNet-V3 Environment Setup Complete
  Python      : 3.14.7
  PyTorch     : 2.13.0+rocm7.2
  HIP / ROCm  : 7.2.53211
  timm        : 1.0.28
  GPU Avail   : True


# Section 2: GPU Detection

In [4]:
# ============================================================
# Section 2: GPU Detection
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "No ROCm GPU detected. The RX 9060 XT should be available."
    )

# Explicitly select the discrete RX 9060 XT
DEVICE = torch.device("cuda:0")

props = torch.cuda.get_device_properties(0)

print("=" * 60)
print("  Hardware Configuration")
print("=" * 60)
print(f"  Backend        : ROCm / HIP")
print(f"  Device         : {torch.cuda.get_device_name(0)}")
print(f"  Architecture   : {props.gcnArchName}")
print(f"  Total Memory   : {props.total_memory / 1024**3:.2f} GiB")
print(f"  Visible GPUs   : {torch.cuda.device_count()}")
print(f"  Active Device  : {DEVICE}")
print("=" * 60)

  Hardware Configuration
  Backend        : ROCm / HIP
  Device         : AMD Radeon RX 9060 XT
  Architecture   : gfx1200
  Total Memory   : 15.92 GiB
  Visible GPUs   : 1
  Active Device  : cuda:0


# Section 3: Kaggle Authentication & Google Drive Mount

In [5]:
# ============================================================
# Section 3: Local Storage & Kaggle Configuration
# ============================================================

from pathlib import Path

# The notebook lives in: OMNet/models/
PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "output_v3"

# Kaggle authentication uses:
# ~/.kaggle/access_token
KAGGLE_TOKEN_PATH = Path.home() / ".kaggle" / "access_token"

if not KAGGLE_TOKEN_PATH.exists():
    raise FileNotFoundError(
        f"Kaggle access token not found: {KAGGLE_TOKEN_PATH}"
    )

# Output directories
GRADCAM_DIR = OUTPUT_DIR / "gradcam"
SCORECAM_DIR = OUTPUT_DIR / "scorecam"
MISCLASSIFIED_DIR = OUTPUT_DIR / "misclassified"
CORRECT_PRED_DIR = OUTPUT_DIR / "correct_predictions"

for directory in [
    OUTPUT_DIR,
    GRADCAM_DIR,
    SCORECAM_DIR,
    MISCLASSIFIED_DIR,
    CORRECT_PRED_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Create fold directories
for fold_idx in range(5):
    (OUTPUT_DIR / f"fold_{fold_idx}").mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 60)
print("  Local Storage Configuration")
print("=" * 60)
print(f"  Project Root : {PROJECT_ROOT}")
print(f"  Output Dir   : {OUTPUT_DIR}")
print(f"  Kaggle Auth  : access_token detected")
print("=" * 60)

  Local Storage Configuration
  Project Root : /home/saviour/00_Workspace/Projects/gnit/OMNet/OMNet
  Output Dir   : /home/saviour/00_Workspace/Projects/gnit/OMNet/OMNet/output_v3
  Kaggle Auth  : access_token detected


# Centralized Configuration

In [19]:
# ============================================================
# Centralized Configuration
# ============================================================

CONFIG = {
    # --- Data ---
    "dataset_id": "saikatd1998/dataset",
    "output_dir": str(OUTPUT_DIR),
    "image_size": 224,
    "batch_size": 16,
    "num_workers": 0,
    "n_splits": 5,
    "magnification_levels": [40, 100, 200, 400],
    "binary_classes": ["benign", "malignant"],
    "subtype_order": ["A", "F", "PT", "TA", "DC", "LC", "MC", "PC"],
    "subtype_to_index": {"A": 0, "F": 1, "PT": 2, "TA": 3, "DC": 4, "LC": 5, "MC": 6, "PC": 7},
    "binary_mapping": {"A": 0, "F": 0, "PT": 0, "TA": 0, "DC": 1, "LC": 1, "MC": 1, "PC": 1},
    "train_resize": 256,
    "train_crop": 224,
    "val_resize": 256,
    "val_crop": 224,
    "stain_aug_prob": 0.5,

    # --- Model ---
    "fusion_dim": 256,
    "magnification_embedding_dim": 64,
    "dropout": 0.3,
    "cnn_backbone": "efficientnet_b0",
    "vit_backbone": "vit_tiny_patch16_224",

    # --- Training ---
    "max_epochs": 30,
    "warmup_epochs": 5,
    "early_stopping_patience": 8,
    "base_lr": 1e-5,
    "head_lr": 1e-4,
    "weight_decay": 1e-4,
    "gradient_clip_norm": 1.0,
    "amp_enabled": False,
    "loss_weights": {"binary": 0.3, "subtype": 0.6, "consistency": 0.1},

    # --- Reproducibility ---
    "seed": 42,
    "smoke_test": False,
}

if CONFIG['smoke_test']:
    CONFIG['n_splits'] = 2
    CONFIG['max_epochs'] = 2
    CONFIG['warmup_epochs'] = 1

set_seed(CONFIG['seed'])

print("=" * 60)
print("  Experiment Configuration (OMNet-V3)")
print("=" * 60)
for k, v in CONFIG.items():
    print(f"  {k:30s}: {v}")
print("=" * 60)

  Experiment Configuration (OMNet-V3)
  dataset_id                    : saikatd1998/dataset
  output_dir                    : /home/saviour/00_Workspace/Projects/gnit/OMNet/OMNet/output_v3
  image_size                    : 224
  batch_size                    : 16
  num_workers                   : 0
  n_splits                      : 5
  magnification_levels          : [40, 100, 200, 400]
  binary_classes                : ['benign', 'malignant']
  subtype_order                 : ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
  subtype_to_index              : {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}
  binary_mapping                : {'A': 0, 'F': 0, 'PT': 0, 'TA': 0, 'DC': 1, 'LC': 1, 'MC': 1, 'PC': 1}
  train_resize                  : 256
  train_crop                    : 224
  val_resize                    : 256
  val_crop                      : 224
  stain_aug_prob                : 0.5
  fusion_dim                    : 256
  magnification_embedding_dim   :

# Section 4: Dataset Download & Discovery via kagglehub

In [40]:
# ============================================================
# Section 4: Dataset Download & Discovery via kagglehub
# ============================================================
import kagglehub

dataset_id = CONFIG['dataset_id']
print(f"[INFO] Downloading dataset '{dataset_id}' directly via kagglehub...")
download_path = kagglehub.dataset_download(dataset_id)
print(f"[INFO] Download path: {download_path}")

local_extract_dir = Path('./data_breakhis')
zip_files = glob.glob(os.path.join(download_path, '**', '*.zip'), recursive=True)

if zip_files:
    os.makedirs(local_extract_dir, exist_ok=True)
    for zf in zip_files:
        print(f"[INFO] Extracting archive {zf} to {local_extract_dir}...")
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(local_extract_dir)
    search_root = local_extract_dir
else:
    search_root = Path(download_path)

print(f"[INFO] Scanning {search_root} for PNG files...")

def parse_breakhis_filename(path: str) -> Optional[dict]:
    filename = os.path.basename(path)
    if not filename.lower().endswith('.png'):
        return None

    fname_no_ext = filename.replace('.png', '')
    parts = fname_no_ext.split('-')

    if len(parts) < 4:
        return None

    try:
        prefix_parts = parts[0].split('_')
        if len(prefix_parts) < 3:
            return None

        raw_class = prefix_parts[1]
        raw_subtype = prefix_parts[2]

        mag_token = parts[-2]
        magnification = int(mag_token)

        if magnification not in CONFIG['magnification_levels']:
            return None

        class_name = 'benign' if raw_class == 'B' else 'malignant'
        subtype = raw_subtype

        if subtype not in CONFIG['subtype_to_index']:
            return None

        case_id = '-'.join(parts[1:-2])
        patient_id = f"{raw_subtype}_{case_id}"
        seq = int(parts[-1])

        return {
            'file_path': path,
            'filename': filename,
            'class_name': class_name,
            'subtype': subtype,
            'subtype_index': CONFIG['subtype_to_index'][subtype],
            'binary_label': CONFIG['binary_mapping'][subtype],
            'magnification': magnification,
            'magnification_index': CONFIG['magnification_levels'].index(magnification),
            'patient_id': patient_id,
            'sequence': seq,
        }
    except Exception:
        return None

records = []
for full_path in Path(search_root).rglob('*.png'):
    parsed = parse_breakhis_filename(str(full_path))
    if parsed is not None:
        records.append(parsed)

assert len(records) > 0, f"[ERROR] No valid BreaKHis images found in {search_root}!"

metadata = pd.DataFrame(records)
print(f"[OK] Discovered {len(metadata)} total images across {metadata['patient_id'].nunique()} patients.")

print("\n--- Distribution by Class ---")
print(metadata['class_name'].value_counts())
print("\n--- Distribution by Subtype ---")
print(metadata['subtype'].value_counts())
print("\n--- Distribution by Magnification ---")
print(metadata['magnification'].value_counts())
print("\n--- Patients per Subtype ---")
print(metadata.groupby('subtype')['patient_id'].nunique())

[INFO] Downloading dataset 'saikatd1998/dataset' directly via kagglehub...
[INFO] Download path: /home/saviour/.cache/kagglehub/datasets/saikatd1998/dataset/versions/1
[INFO] Scanning /home/saviour/.cache/kagglehub/datasets/saikatd1998/dataset/versions/1 for PNG files...
[OK] Discovered 7909 total images across 82 patients.

--- Distribution by Class ---
class_name
malignant    5429
benign       2480
Name: count, dtype: int64

--- Distribution by Subtype ---
subtype
DC    3451
F     1014
MC     792
LC     626
TA     569
PC     560
PT     453
A      444
Name: count, dtype: int64

--- Distribution by Magnification ---
magnification
100    2081
200    2013
40     1995
400    1820
Name: count, dtype: int64

--- Patients per Subtype ---
subtype
A      4
DC    38
F     10
LC     5
MC     9
PC     6
PT     3
TA     7
Name: patient_id, dtype: int64


In [43]:
print("PyTorch:", torch.__version__)
print("ROCm/HIP:", torch.version.hip)
print("CPU cores:", os.cpu_count())

print("\nGPU:")
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")

print("\nCurrent configuration:")
print("batch_size:", CONFIG["batch_size"])
print("num_workers:", CONFIG["num_workers"])
print("image_size:", CONFIG["image_size"])
print("AMP enabled:", CONFIG["amp_enabled"])

PyTorch: 2.13.0+rocm7.2
ROCm/HIP: 7.2.53211
CPU cores: 12

GPU:
AMD Radeon RX 9060 XT
VRAM: 15.92 GiB

Current configuration:
batch_size: 16
num_workers: 0
image_size: 224
AMP enabled: False


In [45]:
CONFIG["n_splits"] = 5

print("n_splits:", CONFIG["n_splits"])

n_splits: 5


# Section 5: Patient-Disjoint Splitting (StratifiedGroupKFold)

In [46]:
# ============================================================
# Section 5: Patient-Disjoint Splitting (StratifiedGroupKFold)
# ============================================================
X = metadata[['file_path', 'subtype', 'patient_id']].copy()
y = metadata['subtype_index'].values
groups = metadata['patient_id'].values

sgkf = StratifiedGroupKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
split_indices = list(sgkf.split(X, y, groups))

print(f"[OK] Created {len(split_indices)} patient-disjoint folds.")

for fold_idx, (train_idx, test_idx) in enumerate(split_indices):
    train_df = metadata.iloc[train_idx]
    test_df = metadata.iloc[test_idx]
    train_p = set(train_df['patient_id'])
    test_p = set(test_df['patient_id'])
    overlap = train_p & test_p
    assert len(overlap) == 0, f"[ERROR] Patient leakage in Fold {fold_idx}: {overlap}"
    print(f"  Fold {fold_idx}: train={len(train_df):4d} ({len(train_p):2d} patients) | test={len(test_df):4d} ({len(test_p):2d} patients)")

# Save split overview
metadata.to_csv(OUTPUT_DIR / 'dataset_metadata.csv', index=False)
print(f"[SAVE] Exported dataset_metadata.csv to {OUTPUT_DIR}")

[OK] Created 5 patient-disjoint folds.
  Fold 0: train=6353 (66 patients) | test=1556 (16 patients)
  Fold 1: train=6380 (66 patients) | test=1529 (16 patients)
  Fold 2: train=6320 (66 patients) | test=1589 (16 patients)
  Fold 3: train=6207 (64 patients) | test=1702 (18 patients)
  Fold 4: train=6376 (66 patients) | test=1533 (16 patients)
[SAVE] Exported dataset_metadata.csv to /home/saviour/00_Workspace/Projects/gnit/OMNet/OMNet/output_v3


In [10]:
# Rare-subtype patient support in each outer test fold

rare_subtypes = ["PT", "A", "LC", "PC", "TA", "MC"]

for fold_idx, (_, test_idx) in enumerate(split_indices):
    test_df = metadata.iloc[test_idx]

    print(f"\nFold {fold_idx}")
    for subtype in rare_subtypes:
        patients = test_df.loc[
            test_df["subtype"] == subtype, "patient_id"
        ].nunique()

        images = (test_df["subtype"] == subtype).sum()

        print(
            f"  {subtype:>2}: "
            f"{patients} patients, "
            f"{images} images"
        )


Fold 0
  PT: 0 patients, 0 images
   A: 1 patients, 130 images
  LC: 1 patients, 201 images
  PC: 2 patients, 127 images
  TA: 1 patients, 66 images
  MC: 2 patients, 160 images

Fold 1
  PT: 1 patients, 60 images
   A: 1 patients, 132 images
  LC: 1 patients, 103 images
  PC: 1 patients, 74 images
  TA: 2 patients, 118 images
  MC: 1 patients, 178 images

Fold 2
  PT: 1 patients, 158 images
   A: 0 patients, 0 images
  LC: 1 patients, 123 images
  PC: 1 patients, 127 images
  TA: 1 patients, 132 images
  MC: 2 patients, 128 images

Fold 3
  PT: 1 patients, 235 images
   A: 1 patients, 121 images
  LC: 1 patients, 74 images
  PC: 1 patients, 90 images
  TA: 2 patients, 129 images
  MC: 2 patients, 152 images

Fold 4
  PT: 0 patients, 0 images
   A: 1 patients, 61 images
  LC: 1 patients, 125 images
  PC: 1 patients, 142 images
  TA: 1 patients, 124 images
  MC: 2 patients, 174 images


In [11]:
from sklearn.metrics import f1_score

print("Per-fold subtype support and macro-F1 feasibility")
print("=" * 60)

for fold_idx, (_, test_idx) in enumerate(split_indices):
    test_df = metadata.iloc[test_idx]

    y_true = test_df["subtype_index"].to_numpy()

    present = sorted(test_df["subtype_index"].unique().tolist())
    missing = sorted(
        set(range(len(CONFIG["subtype_order"]))) - set(present)
    )

    print(f"\nFold {fold_idx}")
    print("Present :", [CONFIG["subtype_order"][i] for i in present])
    print("Missing :", [CONFIG["subtype_order"][i] for i in missing])

Per-fold subtype support and macro-F1 feasibility

Fold 0
Present : ['A', 'F', 'TA', 'DC', 'LC', 'MC', 'PC']
Missing : ['PT']

Fold 1
Present : ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
Missing : []

Fold 2
Present : ['F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
Missing : ['A']

Fold 3
Present : ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
Missing : []

Fold 4
Present : ['A', 'F', 'TA', 'DC', 'LC', 'MC', 'PC']
Missing : ['PT']


# Section 6: Data Augmentations & Stain Perturbation

In [12]:
# ============================================================
# Section 6: Data Augmentations & Stain Perturbation
# ============================================================

def rgb_to_od(img):
    img = img.astype(np.float32) / 255.0
    eps = 1e-6
    img = np.clip(img, eps, 1.0)
    return -np.log(img)

def estimate_he_vectors(od):
    od = od.reshape(-1, 3)
    od = od[~np.any(np.isinf(od) | np.isnan(od), axis=1)]
    if od.shape[0] == 0:
        return np.eye(3)
    _, _, vh = np.linalg.svd(od, full_matrices=False)
    return vh[:2, :]

def macenko_perturb(img: np.ndarray, p: float = 0.5):
    if np.random.rand() > p:
        return img
    arr = np.asarray(img)
    if arr.ndim != 3 or arr.shape[-1] != 3:
        return img
    od = rgb_to_od(arr)
    he_basis = estimate_he_vectors(od)
    stain = np.dot(od.reshape(-1, 3), he_basis.T)
    stain = stain.reshape(arr.shape[0], arr.shape[1], 2)
    if stain.size == 0:
        return img
    rand_factors = np.random.uniform(0.85, 1.15, size=(2,))
    rand_bias = np.random.uniform(-0.05, 0.05, size=(2,))
    perturbed = stain * rand_factors + rand_bias
    recon = np.dot(perturbed.reshape(-1, 2), he_basis).reshape(arr.shape[0], arr.shape[1], 3)
    recon = np.clip(recon, 0.0, 1.0)
    return (recon * 255.0).astype(np.uint8)

print("[OK] Macenko stain perturbation module initialized.")

[OK] Macenko stain perturbation module initialized.


# Section 7: Dataset Class & DataLoaders

In [13]:
# ============================================================
# Section 7: Dataset Class & DataLoaders
# ============================================================

class BreakHisDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split: str = 'train', transform=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return {
            'image': image,
            'binary_label': int(row['binary_label']),
            'subtype_label': int(row['subtype_index']),
            'magnification_index': int(row['magnification_index']),
            'patient_id': row['patient_id'],
            'file_path': row['file_path'],
        }

def build_transforms(split: str):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((CONFIG['train_resize'], CONFIG['train_resize'])),
            transforms.RandomResizedCrop(CONFIG['train_crop'], scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    return transforms.Compose([
        transforms.Resize((CONFIG['val_resize'], CONFIG['val_resize'])),
        transforms.CenterCrop(CONFIG['val_crop']),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

def custom_train_transform(image: Image.Image):
    image = transforms.Resize((CONFIG['train_resize'], CONFIG['train_resize']))(image)
    image = transforms.RandomResizedCrop(CONFIG['train_crop'], scale=(0.8, 1.0))(image)
    image = transforms.RandomHorizontalFlip(p=0.5)(image)
    image = transforms.RandomVerticalFlip(p=0.5)(image)
    image = transforms.RandomRotation(20)(image)

    if np.random.rand() < CONFIG['stain_aug_prob']:
        image_np = np.array(image)
        image_np = macenko_perturb(image_np, p=1.0)
        image = Image.fromarray(image_np.astype(np.uint8))

    image = transforms.ToTensor()(image)
    image = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))(image)
    return image

def build_dataloaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
):
    train_ds = BreakHisDataset(
        train_df,
        "train",
        custom_train_transform,
    )

    val_ds = BreakHisDataset(
        val_df,
        "val",
        build_transforms("val"),
    )

    test_ds = BreakHisDataset(
        test_df,
        "test",
        build_transforms("val"),
    )

    nw = CONFIG["num_workers"]

    loader_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "num_workers": nw,
        "pin_memory": True,
    }

    # These options only apply when worker processes are used.
    if nw > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 2

    train_loader = DataLoader(
        train_ds,
        shuffle=True,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        val_ds,
        shuffle=False,
        **loader_kwargs,
    )

    test_loader = DataLoader(
        test_ds,
        shuffle=False,
        **loader_kwargs,
    )

    return train_loader, val_loader, test_loader


print(
    f"[OK] DataLoaders configured "
    f"(batch_size={CONFIG['batch_size']}, "
    f"num_workers={CONFIG['num_workers']})."
)

[OK] DataLoaders configured (batch_size=16, num_workers=0).


In [14]:
# ============================================================
# Section 7A: DataLoader + GPU Smoke Test
# ============================================================

print("=" * 60)
print("  DataLoader + GPU Smoke Test")
print("=" * 60)

# Temporary validation subset for the smoke test only.
# This does NOT modify train_df or test_df.
smoke_val_df = train_df.sample(
    n=min(256, len(train_df)),
    random_state=CONFIG["seed"],
)

smoke_train_df = train_df.drop(smoke_val_df.index)

# Build loaders
train_loader, val_loader, test_loader = build_dataloaders(
    smoke_train_df,
    smoke_val_df,
    test_df,
)

print(f"Train samples : {len(train_loader.dataset):,}")
print(f"Val samples   : {len(val_loader.dataset):,}")
print(f"Test samples  : {len(test_loader.dataset):,}")

print("\nLoading one training batch...")

batch = next(iter(train_loader))

images = batch["image"]
binary_labels = batch["binary_label"]
subtype_labels = batch["subtype_label"]
magnification_indices = batch["magnification_index"]

print(f"Image batch shape   : {images.shape}")
print(f"Image dtype         : {images.dtype}")
print(f"Binary labels       : {binary_labels.shape}")
print(f"Subtype labels      : {subtype_labels.shape}")
print(f"Magnification       : {magnification_indices.shape}")

print("\nLabel ranges:")
print(
    f"Binary        : "
    f"{binary_labels.min().item()} -> "
    f"{binary_labels.max().item()}"
)
print(
    f"Subtype       : "
    f"{subtype_labels.min().item()} -> "
    f"{subtype_labels.max().item()}"
)
print(
    f"Magnification : "
    f"{magnification_indices.min().item()} -> "
    f"{magnification_indices.max().item()}"
)

print("\nTesting GPU transfer...")

torch.cuda.empty_cache()

images_gpu = images.to(
    DEVICE,
    non_blocking=True,
)

print(f"Active device : {images_gpu.device}")
print(
    f"Allocated VRAM: "
    f"{torch.cuda.memory_allocated(0) / 1024**2:.2f} MiB"
)
print(
    f"Reserved VRAM : "
    f"{torch.cuda.memory_reserved(0) / 1024**2:.2f} MiB"
)

assert images.shape == (
    CONFIG["batch_size"],
    3,
    CONFIG["image_size"],
    CONFIG["image_size"],
)

assert images_gpu.device.type == "cuda"

print("\n" + "=" * 60)
print("  SMOKE TEST PASSED")
print("=" * 60)

  DataLoader + GPU Smoke Test
Train samples : 6,120
Val samples   : 256
Test samples  : 1,533

Loading one training batch...
Image batch shape   : torch.Size([16, 3, 224, 224])
Image dtype         : torch.float32
Binary labels       : torch.Size([16])
Subtype labels      : torch.Size([16])
Magnification       : torch.Size([16])

Label ranges:
Binary        : 0 -> 1
Subtype       : 0 -> 6
Magnification : 0 -> 3

Testing GPU transfer...
Active device : cuda:0
Allocated VRAM: 9.19 MiB
Reserved VRAM : 20.00 MiB

  SMOKE TEST PASSED


# Section 8: OMNet-V3 Architecture (Magnification-Aware Adaptive Fusion)

In [20]:
# ============================================================
# Section 8: OMNet-V3 Architecture
# ============================================================

class MagnificationAwareFusion(nn.Module):
    """
    Magnification-Aware Adaptive Fusion (MAF) Module:
    Dynamically balances CNN local features and ViT global context
    conditioned on magnification scale embedding.
    """
    def __init__(self, feat_dim=256, mag_dim=64):
        super().__init__()
        self.magnification_embedding = nn.Embedding(4, mag_dim)
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * 2 + mag_dim, 1),
            nn.Sigmoid(),
        )
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim + mag_dim, feat_dim),
            nn.GELU(),
            nn.Dropout(CONFIG['dropout']),
            nn.Linear(feat_dim, feat_dim),
        )

    def forward(self, f_cnn, f_vit, magnification_index):
        e_m = self.magnification_embedding(magnification_index)
        concat = torch.cat([f_cnn, f_vit, e_m], dim=-1)
        alpha = self.gate(concat)
        fused = alpha * f_cnn + (1.0 - alpha) * f_vit
        residual = self.mlp(torch.cat([fused, e_m], dim=-1))
        out = fused + residual
        return out, alpha


class OMNetV3(nn.Module):
    """
    OMNet-V3 Dual-Branch Architecture:
      - Branch 1: EfficientNet-B0 (1280-d local morphology)
      - Branch 2: ViT-Tiny/16 (192-d global context)
      - Fusion: Magnification-Aware Adaptive Fusion (256-d)
      - Classification Heads: Binary (2-class) + Subtype (8-class)
    """
    def __init__(self):
        super().__init__()
        self.cnn_backbone = timm.create_model(CONFIG['cnn_backbone'], pretrained=True, num_classes=0)
        self.vit_backbone = timm.create_model(CONFIG['vit_backbone'], pretrained=True, num_classes=0)

        self.cnn_proj = nn.Sequential(
            nn.Linear(1280, CONFIG['fusion_dim']),
            nn.LayerNorm(CONFIG['fusion_dim']),
        )
        self.vit_proj = nn.Sequential(
            nn.Linear(192, CONFIG['fusion_dim']),
            nn.LayerNorm(CONFIG['fusion_dim']),
        )
        self.fusion = MagnificationAwareFusion(
            feat_dim=CONFIG['fusion_dim'],
            mag_dim=CONFIG['magnification_embedding_dim']
        )
        self.binary_head = nn.Linear(CONFIG['fusion_dim'], 2)
        self.subtype_head = nn.Linear(CONFIG['fusion_dim'], 8)

    def forward_cnn(self, x):
        x = self.cnn_backbone.forward_features(x)
        x = self.cnn_backbone.global_pool(x)
        if x.dim() == 4:
            x = x.flatten(1)
        return self.cnn_proj(x)

    def forward_vit(self, x):
        x = self.vit_backbone.forward_features(x)
        if isinstance(x, (tuple, list)):
            x = x[0]
        if x.dim() == 3:
            x = x[:, 0, :]
        return self.vit_proj(x)

    def forward(self, x, magnification_index):
        f_cnn = self.forward_cnn(x)
        f_vit = self.forward_vit(x)
        f_out, alpha = self.fusion(f_cnn, f_vit, magnification_index)
        binary_logits = self.binary_head(f_out)
        subtype_logits = self.subtype_head(f_out)
        return {
            'f_out': f_out,
            'alpha': alpha,
            'binary_logits': binary_logits,
            'subtype_logits': subtype_logits,
        }

# Model sanity check
model_check = OMNetV3().to(DEVICE)
dummy_img = torch.randn(2, 3, 224, 224).to(DEVICE)
dummy_mag = torch.tensor([0, 2]).to(DEVICE)
with torch.no_grad():
    out_check = model_check(dummy_img, dummy_mag)
print("[OK] OMNet-V3 initialized successfully.")
print(f"     Binary logits shape  : {out_check['binary_logits'].shape}")
print(f"     Subtype logits shape : {out_check['subtype_logits'].shape}")
print(f"     Fusion embedding dim : {out_check['f_out'].shape}")
del model_check, dummy_img, dummy_mag, out_check
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

[OK] OMNet-V3 initialized successfully.
     Binary logits shape  : torch.Size([2, 2])
     Subtype logits shape : torch.Size([2, 8])
     Fusion embedding dim : torch.Size([2, 256])


# Section 9: Hierarchical Multi-Task Loss

In [21]:
# ============================================================
# Section 9: Hierarchical Multi-Task Loss
# ============================================================

class HierarchicalLoss(nn.Module):
    def __init__(
        self,
        class_weights_binary: torch.Tensor,
        class_weights_subtype: torch.Tensor,
    ):
        super().__init__()

        self.loss_binary = nn.CrossEntropyLoss(
            weight=class_weights_binary
        )

        self.loss_subtype = nn.CrossEntropyLoss(
            weight=class_weights_subtype
        )

    def forward(
        self,
        binary_logits,
        subtype_logits,
        binary_targets,
        subtype_targets,
    ):
        # Primary supervised losses
        L_binary = self.loss_binary(
            binary_logits,
            binary_targets,
        )

        L_subtype = self.loss_subtype(
            subtype_logits,
            subtype_targets,
        )

        # Convert 8-class subtype probabilities into
        # 2-class benign/malignant probabilities
        subtype_probs = torch.softmax(
            subtype_logits,
            dim=1,
        )

        grouped = torch.stack([
            subtype_probs[:, :4].sum(dim=1),  # A, F, PT, TA
            subtype_probs[:, 4:8].sum(dim=1), # DC, LC, MC, PC
        ], dim=1)

        # Binary branch acts as the reference distribution
        consistency_target = torch.softmax(
            binary_logits,
            dim=1,
        ).detach()

        L_consistency = F.kl_div(
            torch.log(grouped.clamp_min(1e-8)),
            consistency_target,
            reduction='batchmean',
            log_target=False,
        )

        w = CONFIG['loss_weights']

        total = (
            w['binary'] * L_binary
            + w['subtype'] * L_subtype
            + w['consistency'] * L_consistency
        )

        return total, {
            'binary': L_binary,
            'subtype': L_subtype,
            'consistency': L_consistency,
        }


def compute_class_weights(train_df: pd.DataFrame):
    n_total = len(train_df)

    binary_counts = (
        train_df['binary_label']
        .value_counts()
        .sort_index()
    )

    subtype_counts = (
        train_df['subtype_index']
        .value_counts()
        .sort_index()
    )

    binary_weights = torch.tensor([
        n_total / (2 * max(binary_counts.get(i, 1), 1))
        for i in range(2)
    ], dtype=torch.float32)

    subtype_weights = torch.tensor([
        n_total / (8 * max(subtype_counts.get(i, 1), 1))
        for i in range(8)
    ], dtype=torch.float32)

    return binary_weights, subtype_weights


print("[OK] Hierarchical multi-task loss module configured.")

[OK] Hierarchical multi-task loss module configured.


In [22]:
# ============================================================
# Section 9A: Loss Sanity Check
# ============================================================

# Use the existing training dataframe from the smoke-test split
# only to verify that all loss components are finite.

binary_weights, subtype_weights = compute_class_weights(train_df)

criterion_test = HierarchicalLoss(
    binary_weights.to(DEVICE),
    subtype_weights.to(DEVICE),
).to(DEVICE)

# One real batch from the existing DataLoader
batch = next(iter(train_loader))

images = batch["image"].to(DEVICE, non_blocking=True)
binary_targets = batch["binary_label"].to(DEVICE, non_blocking=True)
subtype_targets = batch["subtype_label"].to(DEVICE, non_blocking=True)
magnification_idx = batch["magnification_index"].to(DEVICE, non_blocking=True)

# Fresh model for this test
model_test = OMNetV3().to(DEVICE)
model_test.eval()

with torch.no_grad():
    outputs = model_test(images, magnification_idx)

    total_loss, loss_parts = criterion_test(
        outputs["binary_logits"],
        outputs["subtype_logits"],
        binary_targets,
        subtype_targets,
    )

print("===== LOSS SANITY CHECK =====")
print("Binary loss      :", loss_parts["binary"].item())
print("Subtype loss     :", loss_parts["subtype"].item())
print("Consistency loss :", loss_parts["consistency"].item())
print("Total loss       :", total_loss.item())

print("\nFinite checks:")
print("Binary logits    :", torch.isfinite(outputs["binary_logits"]).all().item())
print("Subtype logits   :", torch.isfinite(outputs["subtype_logits"]).all().item())
print("Binary loss      :", torch.isfinite(loss_parts["binary"]).item())
print("Subtype loss     :", torch.isfinite(loss_parts["subtype"]).item())
print("Consistency loss :", torch.isfinite(loss_parts["consistency"]).item())
print("Total loss       :", torch.isfinite(total_loss).item())

assert torch.isfinite(total_loss)
print("\n[OK] Loss sanity check PASSED.")

del model_test, criterion_test
torch.cuda.empty_cache()

===== LOSS SANITY CHECK =====
Binary loss      : 0.9163091778755188
Subtype loss     : 2.0729057788848877
Consistency loss : 0.06446889042854309
Total loss       : 1.5250831842422485

Finite checks:
Binary logits    : True
Subtype logits   : True
Binary loss      : True
Subtype loss     : True
Consistency loss : True
Total loss       : True

[OK] Loss sanity check PASSED.


In [18]:
# ============================================================
# Section 9B: Single-Batch Backward / AMP Stability Test
# ============================================================

model_test = OMNetV3().to(DEVICE)

binary_weights, subtype_weights = compute_cl[OK] Macenko stain perturbation module initialized.ass_weights(train_df)

criterion_test = HierarchicalLoss(
    binary_weights.to(DEVICE),
    subtype_weights.to(DEVICE),
).to(DEVICE)

optimizer_test = torch.optim.AdamW(
    model_test.parameters(),
    lr=CONFIG["base_lr"],
    weight_decay=CONFIG["weight_decay"],
)

scaler_test = None

model_test.train()

images = batch["image"].to(DEVICE, non_blocking=True)
binary_targets = batch["binary_label"].to(DEVICE, non_blocking=True)
subtype_targets = batch["subtype_label"].to(DEVICE, non_blocking=True)
magnification_idx = batch["magnification_index"].to(
    DEVICE,
    non_blocking=True,
)

optimizer_test.zero_grad(set_to_none=True)

print("===== SINGLE-BATCH AMP/BACKWARD TEST =====")
print("Precision mode: BF16 autocast")

with torch.amp.autocast("cuda",  dtype=torch.bfloat16):
    outputs = model_test(images, magnification_idx)

    loss, loss_parts = criterion_test(
        outputs["binary_logits"],
        outputs["subtype_logits"],
        binary_targets,
        subtype_targets,
    )

print("Forward total loss:", loss.item())

if not torch.isfinite(loss):
    raise RuntimeError(f"Forward loss is non-finite: {loss.item()}")

loss.backward()
# Inspect non-finite gradients
bad_grads = []

for name, param in model_test.named_parameters():
    if param.grad is not None and not torch.isfinite(param.grad).all():
        bad_grads.append(
            (
                name,
                tuple(param.grad.shape),
                float(
                    torch.nan_to_num(
                        param.grad.detach().abs(),
                        nan=0.0,
                        posinf=0.0,
                        neginf=0.0,
                    ).max().item()
                ),
            )
        )

print("Non-finite gradient tensors:", len(bad_grads))

for name, shape, max_abs in bad_grads[:20]:
    print(
        f"  {name:60s} "
        f"shape={shape} "
        f"max_abs={max_abs:.6g}"
    )
print("Backward completed.")


grad_norm = torch.nn.utils.clip_grad_norm_(
    model_test.parameters(),
    CONFIG["gradient_clip_norm"],
)

print("Gradient norm:", float(grad_norm))
print("Gradient norm finite:", torch.isfinite(grad_norm).item())

optimizer_test.step()

# Check parameters after one optimizer step
nonfinite_params = []

for name, param in model_test.named_parameters():
    if param.requires_grad and not torch.isfinite(param).all():
        nonfinite_params.append(name)

print("Non-finite parameters:", len(nonfinite_params))

if nonfinite_params:
    print("First problematic parameter:", nonfinite_params[0])
    raise RuntimeError("Parameters became non-finite after optimizer step.")

print("[OK] Single-batch AMP/BACKWARD test PASSED.")

del model_test, criterion_test, optimizer_test, scaler_test
torch.cuda.empty_cache()

===== SINGLE-BATCH AMP/BACKWARD TEST =====
Precision mode: BF16 autocast
Forward total loss: 1.5054231882095337
Non-finite gradient tensors: 0
Backward completed.
Gradient norm: 8.935375213623047
Gradient norm finite: True
Non-finite parameters: 0
[OK] Single-batch AMP/BACKWARD test PASSED.


In [23]:
# ============================================================
# Section 9C: FP32 Backward Stability Test
# ============================================================

model_fp32 = OMNetV3().to(DEVICE)
criterion_fp32 = HierarchicalLoss(
    binary_weights.to(DEVICE),
    subtype_weights.to(DEVICE),
).to(DEVICE)

optimizer_fp32 = torch.optim.AdamW(
    model_fp32.parameters(),
    lr=CONFIG["base_lr"],
    weight_decay=CONFIG["weight_decay"],
)

model_fp32.train()
optimizer_fp32.zero_grad(set_to_none=True)

print("===== FP32 BACKWARD TEST =====")

# No autocast, no GradScaler
outputs_fp32 = model_fp32(
    images.float(),
    magnification_idx,
)

loss_fp32, loss_parts_fp32 = criterion_fp32(
    outputs_fp32["binary_logits"],
    outputs_fp32["subtype_logits"],
    binary_targets,
    subtype_targets,
)

print("Forward loss:", loss_fp32.item())

if not torch.isfinite(loss_fp32):
    raise RuntimeError(
        f"FP32 forward loss is non-finite: {loss_fp32.item()}"
    )

loss_fp32.backward()

nonfinite_grads = []

for name, param in model_fp32.named_parameters():
    if param.requires_grad and param.grad is not None:
        if not torch.isfinite(param.grad).all():
            nonfinite_grads.append(name)

grad_norm_fp32 = torch.nn.utils.clip_grad_norm_(
    model_fp32.parameters(),
    CONFIG["gradient_clip_norm"],
)

print("Gradient norm:", float(grad_norm_fp32))
print(
    "Gradient norm finite:",
    torch.isfinite(grad_norm_fp32).item()
)
print(
    "Non-finite gradient tensors:",
    len(nonfinite_grads)
)

if nonfinite_grads:
    print("First problematic gradient:", nonfinite_grads[0])

if not nonfinite_grads and torch.isfinite(grad_norm_fp32):
    optimizer_fp32.step()
    print("[OK] FP32 backward/optimizer test PASSED.")
else:
    raise RuntimeError(
        "FP32 backward produced non-finite gradients."
    )

del model_fp32, criterion_fp32, optimizer_fp32
torch.cuda.empty_cache()

===== FP32 BACKWARD TEST =====
Forward loss: 1.4847244024276733
Gradient norm: 7.0938239097595215
Gradient norm finite: True
Non-finite gradient tensors: 0
[OK] FP32 backward/optimizer test PASSED.


In [32]:
# ============================================================
# Section 9D: BF16 AMP Backward Stability Test
# ============================================================

print("===== BF16 AMP BACKWARD TEST =====")

print(
    "BF16 supported:",
    torch.cuda.is_bf16_supported()
)

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "This PyTorch/ROCm device does not report BF16 support."
    )

model_bf16 = OMNetV3().to(DEVICE)

criterion_bf16 = HierarchicalLoss(
    binary_weights.to(DEVICE),
    subtype_weights.to(DEVICE),
).to(DEVICE)

optimizer_bf16 = torch.optim.AdamW(
    model_bf16.parameters(),
    lr=CONFIG["base_lr"],
    weight_decay=CONFIG["weight_decay"],
)

model_bf16.train()
optimizer_bf16.zero_grad(set_to_none=True)

with torch.amp.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
):
    outputs_bf16 = model_bf16(
        images,
        magnification_idx,
    )

    loss_bf16, loss_parts_bf16 = criterion_bf16(
        outputs_bf16["binary_logits"],
        outputs_bf16["subtype_logits"],
        binary_targets,
        subtype_targets,
    )

print(
    "Forward loss:",
    loss_bf16.item()
)

print(
    "Binary loss:",
    loss_parts_bf16["binary"].item()
)

print(
    "Subtype loss:",
    loss_parts_bf16["subtype"].item()
)

print(
    "Consistency loss:",
    loss_parts_bf16["consistency"].item()
)

if not torch.isfinite(loss_bf16):
    raise RuntimeError(
        "BF16 forward loss is non-finite."
    )

# No GradScaler for this diagnostic.
loss_bf16.backward()

nonfinite_bf16_grads = []

for name, param in model_bf16.named_parameters():
    if param.requires_grad and param.grad is not None:
        if not torch.isfinite(param.grad).all():
            nonfinite_bf16_grads.append(name)

grad_norm_bf16 = torch.nn.utils.clip_grad_norm_(
    model_bf16.parameters(),
    CONFIG["gradient_clip_norm"],
)

print(
    "Gradient norm:",
    float(grad_norm_bf16)
)

print(
    "Gradient norm finite:",
    torch.isfinite(grad_norm_bf16).item()
)

print(
    "Non-finite gradient tensors:",
    len(nonfinite_bf16_grads)
)

if nonfinite_bf16_grads:
    print(
        "First problematic gradient:",
        nonfinite_bf16_grads[0]
    )

if (
    not nonfinite_bf16_grads
    and torch.isfinite(grad_norm_bf16)
):
    optimizer_bf16.step()
    print("[OK] BF16 AMP backward test PASSED.")
else:
    raise RuntimeError(
        "BF16 backward produced non-finite gradients."
    )

del model_bf16, criterion_bf16, optimizer_bf16
torch.cuda.empty_cache()

===== BF16 AMP BACKWARD TEST =====
BF16 supported: True
Forward loss: 1.488523006439209
Binary loss: 0.8270361423492432
Subtype loss: 2.0640666484832764
Consistency loss: 0.019720900803804398
Gradient norm: 11.003002166748047
Gradient norm finite: True
Non-finite gradient tensors: 0
[OK] BF16 AMP backward test PASSED.


In [ ]:
CONFIG['n_splits'] = 1
CONFIG['max_epochs'] = 6

print("Temporary BF16 verification:")
print("n_splits  :", CONFIG['n_splits'])
print("max_epochs:", CONFIG['max_epochs'])
print("AMP       :", CONFIG['amp_enabled'])

In [47]:
CONFIG["n_splits"] = 5
CONFIG["max_epochs"] = 30

print("Full V3 configuration:")
print("n_splits :", CONFIG["n_splits"])
print("max_epochs:", CONFIG["max_epochs"])
print("AMP      :", CONFIG["amp_enabled"])

Full V3 configuration:
n_splits : 5
max_epochs: 30
AMP      : False


# Section 10: Training Engine (Modern torch.amp)

In [24]:
# ============================================================
# Section 10: Training Engine (FP32 / ROCm)
# ============================================================

def set_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad = flag


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0

    preds_binary = []
    preds_subtype = []

    labels_binary = []
    labels_subtype = []

    # --------------------------------------------------------
    # IMPORTANT:
    # Local ROCm FP32 training is used deliberately.
    #
    # Repeated BF16 backward on this RX 9060 XT /
    # PyTorch 2.13.0+rocm7.2 stack reproducibly triggered
    # an amdgpu GPUVM page fault.
    #
    # FP32 repeated dual-backbone training was stable.
    # --------------------------------------------------------
    use_amp = False

    for batch_idx, batch in enumerate(
        tqdm(loader, leave=False)
    ):
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        binary_targets = batch["binary_label"].to(
            device,
            non_blocking=True,
        )

        subtype_targets = batch["subtype_label"].to(
            device,
            non_blocking=True,
        )

        magnification_idx = batch[
            "magnification_index"
        ].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        # ----------------------------------------------------
        # FP32 forward
        # ----------------------------------------------------
        outputs = model(
            images,
            magnification_idx,
        )

        loss, loss_parts = criterion(
            outputs["binary_logits"],
            outputs["subtype_logits"],
            binary_targets,
            subtype_targets,
        )

        # ----------------------------------------------------
        # Forward numerical safety
        # ----------------------------------------------------
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss detected at batch "
                f"{batch_idx}: {loss.item()}"
            )

        if not torch.isfinite(
            outputs["binary_logits"]
        ).all():
            raise RuntimeError(
                f"Non-finite binary logits at batch "
                f"{batch_idx}."
            )

        if not torch.isfinite(
            outputs["subtype_logits"]
        ).all():
            raise RuntimeError(
                f"Non-finite subtype logits at batch "
                f"{batch_idx}."
            )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------
        loss.backward()

        # ----------------------------------------------------
        # Gradient safety
        # ----------------------------------------------------
        nonfinite_gradients = []

        for name, param in model.named_parameters():
            if (
                param.requires_grad
                and param.grad is not None
                and not torch.isfinite(param.grad).all()
            ):
                nonfinite_gradients.append(name)

        if nonfinite_gradients:
            raise RuntimeError(
                "Non-finite gradients detected at "
                f"batch {batch_idx}. "
                f"First parameter: {nonfinite_gradients[0]}"
            )

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG["gradient_clip_norm"],
        )

        if not torch.isfinite(grad_norm):
            raise RuntimeError(
                f"Non-finite gradient norm at batch "
                f"{batch_idx}: {float(grad_norm)}"
            )

        optimizer.step()

        # ----------------------------------------------------
        # Safety check after optimizer update
        # ----------------------------------------------------
        nonfinite_parameters = []

        for name, param in model.named_parameters():
            if (
                param.requires_grad
                and not torch.isfinite(param).all()
            ):
                nonfinite_parameters.append(name)

        if nonfinite_parameters:
            raise RuntimeError(
                "Non-finite parameters detected after "
                f"optimizer step at batch {batch_idx}. "
                f"First parameter: {nonfinite_parameters[0]}"
            )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------
        running_loss += (
            loss.detach().item()
            * images.size(0)
        )

        preds_binary.append(
            torch.argmax(
                outputs["binary_logits"],
                dim=1,
            ).detach().cpu()
        )

        preds_subtype.append(
            torch.argmax(
                outputs["subtype_logits"],
                dim=1,
            ).detach().cpu()
        )

        labels_binary.append(
            binary_targets.detach().cpu()
        )

        labels_subtype.append(
            subtype_targets.detach().cpu()
        )

    # --------------------------------------------------------
    # Epoch metrics
    # --------------------------------------------------------
    epoch_loss = running_loss / max(
        len(loader.dataset),
        1,
    )

    binary_pred = torch.cat(
        preds_binary
    ).numpy()

    subtype_pred = torch.cat(
        preds_subtype
    ).numpy()

    binary_true = torch.cat(
        labels_binary
    ).numpy()

    subtype_true = torch.cat(
        labels_subtype
    ).numpy()

    return {
        "loss": epoch_loss,

        "binary_accuracy": accuracy_score(
            binary_true,
            binary_pred,
        ),

        "subtype_accuracy": accuracy_score(
            subtype_true,
            subtype_pred,
        ),

        "subtype_macro_f1": f1_score(
            subtype_true,
            subtype_pred,
            average="macro",
            zero_division=np.nan,
        ),
    }


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    loader,
    device,
):
    model.eval()

    logits_binary = []
    logits_subtype = []

    labels_binary = []
    labels_subtype = []

    patient_ids = []
    files = []

    # --------------------------------------------------------
    # Evaluation is also FP32 for consistency with training.
    # --------------------------------------------------------
    use_amp = False

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        magnification_idx = batch[
            "magnification_index"
        ].to(
            device,
            non_blocking=True,
        )

        # ----------------------------------------------------
        # FP32 evaluation
        # ----------------------------------------------------
        outputs = model(
            images,
            magnification_idx,
        )

        # ----------------------------------------------------
        # Numerical safety
        # ----------------------------------------------------
        if not torch.isfinite(
            outputs["binary_logits"]
        ).all():
            raise RuntimeError(
                "Non-finite binary logits encountered "
                "during evaluation."
            )

        if not torch.isfinite(
            outputs["subtype_logits"]
        ).all():
            raise RuntimeError(
                "Non-finite subtype logits encountered "
                "during evaluation."
            )

        logits_binary.append(
            outputs["binary_logits"]
            .float()
            .cpu()
        )

        logits_subtype.append(
            outputs["subtype_logits"]
            .float()
            .cpu()
        )

        labels_binary.append(
            batch["binary_label"]
        )

        labels_subtype.append(
            batch["subtype_label"]
        )

        patient_ids.extend(
            batch["patient_id"]
        )

        files.extend(
            batch["file_path"]
        )

    # --------------------------------------------------------
    # Convert accumulated logits
    # --------------------------------------------------------
    binary_logits_cpu = torch.cat(
        logits_binary
    )

    subtype_logits_cpu = torch.cat(
        logits_subtype
    )

    binary_probs = torch.softmax(
        binary_logits_cpu,
        dim=1,
    ).numpy()

    subtype_probs = torch.softmax(
        subtype_logits_cpu,
        dim=1,
    ).numpy()

    binary_preds = np.argmax(
        binary_probs,
        axis=1,
    )

    subtype_preds = np.argmax(
        subtype_probs,
        axis=1,
    )

    binary_true = torch.cat(
        labels_binary
    ).numpy()

    subtype_true = torch.cat(
        labels_subtype
    ).numpy()

    return {
        "binary_accuracy": accuracy_score(
            binary_true,
            binary_preds,
        ),

        "binary_macro_f1": f1_score(
            binary_true,
            binary_preds,
            average="macro",
            zero_division=0,
        ),

        "binary_balanced_accuracy":
            balanced_accuracy_score(
                binary_true,
                binary_preds,
            ),

        "binary_mcc": matthews_corrcoef(
            binary_true,
            binary_preds,
        ),

        "subtype_accuracy": accuracy_score(
            subtype_true,
            subtype_preds,
        ),

        "subtype_macro_f1": f1_score(
            subtype_true,
            subtype_preds,
            average="macro",
            zero_division=np.nan,
        ),

        "subtype_weighted_f1": f1_score(
            subtype_true,
            subtype_preds,
            average="weighted",
            zero_division=0,
        ),

        "subtype_balanced_accuracy":
            balanced_accuracy_score(
                subtype_true,
                subtype_preds,
            ),

        "subtype_mcc": matthews_corrcoef(
            subtype_true,
            subtype_preds,
        ),

        "binary_confusion_matrix":
            confusion_matrix(
                binary_true,
                binary_preds,
            ).tolist(),

        "subtype_confusion_matrix":
            confusion_matrix(
                subtype_true,
                subtype_preds,
            ).tolist(),

        "subtype_probs":
            subtype_probs.tolist(),

        "binary_probs":
            binary_probs.tolist(),

        "subtype_preds":
            subtype_preds.tolist(),

        "binary_preds":
            binary_preds.tolist(),

        "subtype_true":
            subtype_true.tolist(),

        "binary_true":
            binary_true.tolist(),

        "patient_ids":
            patient_ids,

        "files":
            files,
    }


print(
    "[OK] Training & evaluation functions ready "
    "(FP32)."
)

[OK] Training & evaluation functions ready (FP32).


In [31]:
print("===== CURRENT TRAINING CONFIG =====")

for key in [
    'n_splits',
    'max_epochs',
    'warmup_epochs',
    'early_stopping_patience',
    'batch_size',
    'base_lr',
    'head_lr',
    'weight_decay',
    'gradient_clip_norm',
    'amp_enabled',
    'smoke_test',
]:
    print(f"{key}: {CONFIG[key]}")

===== CURRENT TRAINING CONFIG =====
n_splits: 1
max_epochs: 6
warmup_epochs: 5
early_stopping_patience: 8
batch_size: 16
base_lr: 1e-05
head_lr: 0.0001
weight_decay: 0.0001
gradient_clip_norm: 1.0
amp_enabled: False
smoke_test: False


In [32]:
CONFIG["n_splits"] = 1
CONFIG["max_epochs"] = 6

print("Temporary pilot configuration:")
print("n_splits :", CONFIG["n_splits"])
print("max_epochs:", CONFIG["max_epochs"])
print("AMP      :", CONFIG["amp_enabled"])

Temporary pilot configuration:
n_splits : 1
max_epochs: 6
AMP      : False


# Section 11: 5-Fold Cross-Validation Execution

In [ ]:
# ============================================================
# Section 11: 5-Fold Cross-Validation Execution
# ============================================================

import time


def run_fold(
    fold_idx: int,
    full_df: pd.DataFrame,
):
    fold_output_dir = (
        OUTPUT_DIR / f'fold_{fold_idx}'
    )

    fold_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # --------------------------------------------------------
    # Patient-disjoint train/test split
    # --------------------------------------------------------
    train_idx, test_idx = split_indices[fold_idx]

    train_df_full = (
        full_df.iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    test_df = (
        full_df.iloc[test_idx]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Patient-disjoint validation split
    # --------------------------------------------------------
    train_patients = set(
        train_df_full['patient_id']
    )

    val_sample_count = max(
        1,
        int(0.10 * len(train_patients)),
    )

    val_patients = set(
        pd.Series(
            list(train_patients)
        ).sample(
            val_sample_count,
            random_state=(
                CONFIG['seed'] + fold_idx
            ),
        )
    )

    val_df = (
        train_df_full[
            train_df_full['patient_id'].isin(
                val_patients
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    train_df = (
        train_df_full[
            ~train_df_full['patient_id'].isin(
                val_patients
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------
    binary_weights, subtype_weights = (
        compute_class_weights(train_df)
    )

    criterion = HierarchicalLoss(
        binary_weights.to(DEVICE),
        subtype_weights.to(DEVICE),
    ).to(DEVICE)

    # --------------------------------------------------------
    # DataLoaders
    # --------------------------------------------------------
    train_loader, val_loader, test_loader = (
        build_dataloaders(
            train_df,
            val_df,
            test_df,
        )
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    model = OMNetV3().to(DEVICE)

    # --------------------------------------------------------
    # Parameter groups
    # --------------------------------------------------------
    base_params = (
        list(model.cnn_backbone.parameters())
        + list(model.vit_backbone.parameters())
    )

    head_params = (
        list(model.cnn_proj.parameters())
        + list(model.vit_proj.parameters())
        + list(model.fusion.parameters())
        + list(model.binary_head.parameters())
        + list(model.subtype_head.parameters())
    )

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    optimizer = torch.optim.AdamW(
        [
            {
                'params': base_params,
                'lr': CONFIG['base_lr'],
            },
            {
                'params': head_params,
                'lr': CONFIG['head_lr'],
            },
        ],
        weight_decay=CONFIG['weight_decay'],
    )

    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=CONFIG['max_epochs'],
        )
    )

    best_val_f1 = -1.0
    best_state = None
    best_epoch = None
    history = []
    patience_counter = 0

    print(
        f"\n--- Fold {fold_idx}: "
        f"Train={len(train_df)} imgs "
        f"({train_df['patient_id'].nunique()} pats) | "
        f"Val={len(val_df)} imgs "
        f"({val_df['patient_id'].nunique()} pats) | "
        f"Test={len(test_df)} imgs "
        f"({test_df['patient_id'].nunique()} pats) ---"
    )

    # --------------------------------------------------------
    # Epoch loop
    # --------------------------------------------------------
    for epoch in range(
        1,
        CONFIG['max_epochs'] + 1,
    ):
        epoch_start = time.time()

        # ----------------------------------------------------
        # Progressive unfreezing
        # ----------------------------------------------------
        if epoch <= CONFIG['warmup_epochs']:
            # Freeze pretrained backbones
            set_requires_grad(
                model.cnn_backbone,
                False,
            )

            set_requires_grad(
                model.vit_backbone,
                False,
            )

            # Keep backbone normalization/statistics frozen too.
            model.cnn_backbone.eval()
            model.vit_backbone.eval()

        else:
            # Unfreeze pretrained backbones
            set_requires_grad(
                model.cnn_backbone,
                True,
            )

            set_requires_grad(
                model.vit_backbone,
                True,
            )

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            DEVICE,
        )

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------
        val_metrics = evaluate_model(
            model,
            val_loader,
            DEVICE,
        )

        scheduler.step()

        epoch_time = (
            time.time() - epoch_start
        )

        record = {
            'epoch': epoch,
            'train_loss': round(
                train_metrics['loss'],
                4,
            ),
            'train_subtype_f1': round(
                train_metrics[
                    'subtype_macro_f1'
                ],
                4,
            ),
            'val_binary_f1': round(
                val_metrics[
                    'binary_macro_f1'
                ],
                4,
            ),
            'val_subtype_macro_f1': round(
                val_metrics[
                    'subtype_macro_f1'
                ],
                4,
            ),
            'val_subtype_acc': round(
                val_metrics[
                    'subtype_accuracy'
                ],
                4,
            ),
        }

        history.append(record)

        print(
            f"  Epoch {epoch:2d} | "
            f"{epoch_time:5.1f}s | "
            f"Train Loss: "
            f"{record['train_loss']:.4f} | "
            f"Val Subtype F1: "
            f"{record['val_subtype_macro_f1']:.4f} | "
            f"Val Bin F1: "
            f"{record['val_binary_f1']:.4f}"
        )

        # ----------------------------------------------------
        # Checkpoint best validation model
        # ----------------------------------------------------
        if (
            val_metrics['subtype_macro_f1']
            > best_val_f1
        ):
            best_val_f1 = (
                val_metrics[
                    'subtype_macro_f1'
                ]
            )

            # --------------------------------------------------------
            # Save best model + reproducibility metadata
            # --------------------------------------------------------
            
            best_epoch = epoch
            
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            
            # Keep the original weight-only checkpoint format for
            # compatibility with existing analysis code.
            torch.save(
                best_state,
                fold_output_dir / "best_model.pth",
            )
            
            # Save metadata separately so the checkpoint remains
            # simple while the experiment remains reproducible.
            config_snapshot = {}
            
            for key, value in CONFIG.items():
                if isinstance(value, Path):
                    config_snapshot[key] = str(value)
                else:
                    config_snapshot[key] = value
            
            checkpoint_metadata = {
                "fold": int(fold_idx),
                "best_epoch": int(best_epoch),
                "best_val_subtype_macro_f1": float(best_val_f1),
                "training_precision": "FP32",
                "amp_enabled": bool(CONFIG["amp_enabled"]),
                "device": str(DEVICE),
                "gpu_name": torch.cuda.get_device_name(0)
                    if torch.cuda.is_available()
                    else "CPU",
                "torch_version": torch.__version__,
                "hip_version": torch.version.hip,
                "config": config_snapshot,
            }
            
            with open(
                fold_output_dir / "checkpoint_metadata.json",
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    checkpoint_metadata,
                    f,
                    indent=2,
                )
            
            print(
                f"  [SAVE] Best checkpoint updated: "
                f"epoch={best_epoch}, "
                f"val_subtype_f1={best_val_f1:.4f}"
            )
            
            patience_counter = 0

        else:
            patience_counter += 1

            if (
                patience_counter
                >= CONFIG[
                    'early_stopping_patience'
                ]
                and not CONFIG['smoke_test']
            ):
                print(
                    f"  [INFO] Early stopping "
                    f"triggered at epoch {epoch}."
                )
                break

    # --------------------------------------------------------
    # Save training history
    # --------------------------------------------------------
    with open(
        fold_output_dir
        / 'training_history.json',
        'w',
    ) as f:
        json.dump(
            history,
            f,
            indent=2,
        )

    # --------------------------------------------------------
    # Evaluate best model on held-out test set
    # --------------------------------------------------------
    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    model.to(DEVICE)

    test_metrics = evaluate_model(
        model,
        test_loader,
        DEVICE,
    )

    with open(
        fold_output_dir
        / 'test_metrics.json',
        'w',
    ) as f:
        json.dump(
            test_metrics,
            f,
            indent=2,
        )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------
    del (
        model,
        optimizer,
        train_loader,
        val_loader,
        test_loader,
        criterion,
    )

    gc.collect()

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    return test_metrics, history


# ============================================================
# Run all folds
# ============================================================

all_fold_results = []

print("=" * 60)
print("  OMNet-V3 Patient-Disjoint Cross-Validation")
print("=" * 60)
print(
    f"  Folds         : {CONFIG['n_splits']}"
)
print(
    f"  Max epochs    : {CONFIG['max_epochs']}"
)
print(
    f"  Batch size    : {CONFIG['batch_size']}"
)
print(
    f"  Base LR       : {CONFIG['base_lr']}"
)
print(
    f"  Head LR       : {CONFIG['head_lr']}"
)
print(
    f"  AMP           : {CONFIG['amp_enabled']}"
)
print(
    "  Precision     : FP32"
)
print(
    f"  Device        : "
    f"{torch.cuda.get_device_name(0)}"
)
print("=" * 60)


for fold_idx in range(
    CONFIG['n_splits']
):
    print(
        f"\n============================================================"
    )
    print(
        f"  Running Patient-Disjoint "
        f"Fold {fold_idx + 1}/{CONFIG['n_splits']}"
    )
    print(
        f"============================================================"
    )

    test_metrics, history = run_fold(
        fold_idx,
        metadata,
    )

    all_fold_results.append(
        {
            'fold_idx': fold_idx,
            'test_metrics': test_metrics,
            'history': history,
        }
    )

    print(
        f"[OK] Fold {fold_idx} finished. "
        f"Test Subtype Macro-F1: "
        f"{test_metrics['subtype_macro_f1']:.4f} | "
        f"Binary Acc: "
        f"{test_metrics['binary_accuracy']:.4f}"
    )

  OMNet-V3 Patient-Disjoint Cross-Validation
  Folds         : 5
  Max epochs    : 30
  Batch size    : 16
  Base LR       : 1e-05
  Head LR       : 0.0001
  AMP           : False
  Precision     : FP32
  Device        : AMD Radeon RX 9060 XT

  Running Patient-Disjoint Fold 1/5

--- Fold 0: Train=5829 imgs (60 pats) | Val=524 imgs (6 pats) | Test=1556 imgs (16 pats) ---


  Epoch  1 |  99.1s | Train Loss: 1.1566 | Val Subtype F1: 0.1274 | Val Bin F1: 0.6220
  [SAVE] Best checkpoint updated: epoch=1, val_subtype_f1=0.1274


  Epoch  2 |  98.7s | Train Loss: 0.9168 | Val Subtype F1: 0.1178 | Val Bin F1: 0.6748


  Epoch  3 |  98.0s | Train Loss: 0.7987 | Val Subtype F1: 0.1142 | Val Bin F1: 0.6863


  Epoch  4 |  98.2s | Train Loss: 0.7293 | Val Subtype F1: 0.1000 | Val Bin F1: 0.6794


...

  Epoch 18 | 178.0s | Train Loss: 0.2048 | Val Subtype F1: 0.0835 | Val Bin F1: 0.4874


  Epoch 19 | 190.9s | Train Loss: 0.1883 | Val Subtype F1: 0.0970 | Val Bin F1: 0.4895


  Epoch 20 | 191.8s | Train Loss: 0.1928 | Val Subtype F1: 0.0873 | Val Bin F1: 0.4870


  Epoch 21 | 191.3s | Train Loss: 0.1864 | Val Subtype F1: 0.0824 | Val Bin F1: 0.4883


  1%|█▏                                                                                                            | 4/360 [00:02<03:04,  1.93it/s]

# Section 12: Cross-Fold Results Aggregation

In [ ]:
# ============================================================
# Section 12: Cross-Fold Results Aggregation
# ============================================================

def aggregate_results(results):
    metric_keys = [
        'binary_accuracy', 'binary_macro_f1', 'binary_balanced_accuracy', 'binary_mcc',
        'subtype_accuracy', 'subtype_macro_f1', 'subtype_weighted_f1', 'subtype_balanced_accuracy', 'subtype_mcc'
    ]
    summary = {}
    for key in metric_keys:
        values = [r['test_metrics'][key] for r in results]
        values = np.array(values, dtype=float)
        summary[key] = {
            'mean': float(values.mean()),
            'std': float(values.std()),
            'min': float(values.min()),
            'max': float(values.max()),
        }
    return summary

aggregated_summary = aggregate_results(all_fold_results)

print("=" * 75)
print("  OMNet-V3 Cross-Fold Aggregated Performance (Mean +/- Std across 5 Folds)")
print("=" * 75)
for k, stats in aggregated_summary.items():
    print(f"  {k:<30s}: {stats['mean']:.4f} +/- {stats['std']:.4f} (range: [{stats['min']:.4f}, {stats['max']:.4f}])")
print("=" * 75)

with open(OUTPUT_DIR / 'aggregated_results.json', 'w') as f:
    json.dump(aggregated_summary, f, indent=2)
print(f"[SAVE] Exported aggregated_results.json to {OUTPUT_DIR}")

# Section 13: Confusion Matrices (Binary & 8-Subtype)

In [ ]:
# ============================================================
# Section 13: Confusion Matrices (Binary & 8-Subtype)
# ============================================================

def plot_confusion_matrix(cm, labels, title, fname):
    fig, ax = plt.subplots(figsize=(7, 6))
    cm_arr = np.array(cm)
    cm_norm = cm_arr.astype(float) / np.maximum(cm_arr.sum(axis=1, keepdims=True), 1)
    labels_annot = np.array([f'{c}\n({p:.1%})' for c, p in zip(cm_arr.flatten(), cm_norm.flatten())]).reshape(cm_arr.shape)
    sns.heatmap(cm_arr, annot=labels_annot, fmt='', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.show()

# Aggregate confusion matrices across folds
cm_binary_total = np.zeros((2, 2), dtype=int)
cm_subtype_total = np.zeros((8, 8), dtype=int)

for res in all_fold_results:
    cm_binary_total += np.array(res['test_metrics']['binary_confusion_matrix'])
    cm_subtype_total += np.array(res['test_metrics']['subtype_confusion_matrix'])

plot_confusion_matrix(cm_binary_total, CONFIG['binary_classes'], 'Aggregated Binary Confusion Matrix (All Folds)', str(OUTPUT_DIR / 'confusion_matrix_binary.png'))
plot_confusion_matrix(cm_subtype_total, CONFIG['subtype_order'], 'Aggregated 8-Subtype Confusion Matrix (All Folds)', str(OUTPUT_DIR / 'confusion_matrix_8class.png'))

# Section 14: ROC & Precision-Recall Curves

In [ ]:
# ============================================================
# Section 14: ROC & Precision-Recall Curves
# ============================================================

all_bin_probs, all_bin_trues = [], []
all_sub_probs, all_sub_trues = [], []

for res in all_fold_results:
    all_bin_probs.extend([p[1] for p in res['test_metrics']['binary_probs']])
    all_bin_trues.extend(res['test_metrics']['binary_true'])
    all_sub_probs.extend(res['test_metrics']['subtype_probs'])
    all_sub_trues.extend(res['test_metrics']['subtype_true'])

all_bin_probs = np.array(all_bin_probs)
all_bin_trues = np.array(all_bin_trues)
all_sub_probs = np.array(all_sub_probs)
all_sub_trues = np.array(all_sub_trues)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Binary ROC
fpr, tpr, _ = roc_curve(all_bin_trues, all_bin_probs)
bin_auc = roc_auc_score(all_bin_trues, all_bin_probs)
axes[0].plot(fpr, tpr, label=f'Binary (AUC={bin_auc:.4f})', color='#e74c3c', linewidth=2.5)
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_title('Binary ROC Curve (Aggregated Folds)', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# Subtype OvR ROC Curves
for i, subtype in enumerate(CONFIG['subtype_order']):
    sub_true_binary = (all_sub_trues == i).astype(int)
    if sub_true_binary.sum() > 0:
        sub_fpr, sub_tpr, _ = roc_curve(sub_true_binary, all_sub_probs[:, i])
        sub_auc = roc_auc_score(sub_true_binary, all_sub_probs[:, i])
        axes[1].plot(sub_fpr, sub_tpr, label=f'{subtype} (AUC={sub_auc:.3f})')

axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('Subtype OvR ROC Curves', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
save_figure(fig, 'roc_curves.png')
plt.show()

# Section 15: Fusion Gate Analysis across Magnifications

In [ ]:
# ============================================================
# Section 15: Fusion Gate Analysis across Magnifications
# ============================================================

best_fold_model = OMNetV3().to(DEVICE)
best_fold_ckpt = torch.load(OUTPUT_DIR / 'fold_0' / 'best_model.pth', map_location=DEVICE, weights_only=False)
best_fold_model.load_state_dict(best_fold_ckpt)
best_fold_model.eval()

_, _, test_loader_f0 = build_dataloaders(
    metadata.iloc[split_indices[0][0]],
    metadata.iloc[split_indices[0][0][:10]],
    metadata.iloc[split_indices[0][1]]
)

alpha_by_mag = {m: [] for m in CONFIG['magnification_levels']}
with torch.no_grad():
    for batch in test_loader_f0:
        images = batch['image'].to(DEVICE)
        mag_idx = batch['magnification_index'].to(DEVICE)
        outputs = best_fold_model(images, mag_idx)
        alphas = outputs['alpha'].detach().cpu().flatten().numpy()
        for i, m_idx in enumerate(batch['magnification_index'].numpy()):
            mag_val = CONFIG['magnification_levels'][m_idx]
            alpha_by_mag[mag_val].append(alphas[i])

print("=" * 60)
print("  Fusion Gate (Alpha) Distribution by Magnification")
print("=" * 60)
for mag, alphas in alpha_by_mag.items():
    if alphas:
        print(f"  {mag:3d}X: mean alpha = {np.mean(alphas):.4f} +/- {np.std(alphas):.4f} (CNN vs ViT balance)")
print("=" * 60)

fig, ax = plt.subplots(figsize=(8, 5))
data_for_plot = [[mag, a] for mag, alphas in alpha_by_mag.items() for a in alphas]
df_gate = pd.DataFrame(data_for_plot, columns=['Magnification', 'Alpha'])
sns.boxplot(data=df_gate, x='Magnification', y='Alpha', ax=ax, palette='Blues')
ax.set_title('Adaptive Fusion Gate Alpha across Magnification Levels', fontweight='bold')
ax.set_ylabel('Alpha (Weight on CNN branch)')
plt.tight_layout()
save_figure(fig, 'fusion_gate_analysis.png')
plt.show()

# Section 16: t-SNE Embedding Visualization & Grad-CAM

In [ ]:
# ============================================================
# Section 16: t-SNE Embedding Visualization & Grad-CAM
# ============================================================

all_features, all_sub_lbls = [], []
with torch.no_grad():
    for batch in test_loader_f0:
        images = batch['image'].to(DEVICE)
        mag_idx = batch['magnification_index'].to(DEVICE)
        outputs = best_fold_model(images, mag_idx)
        all_features.append(outputs['f_out'].cpu().numpy())
        all_sub_lbls.extend(batch['subtype_label'].numpy())

all_features = np.concatenate(all_features, axis=0)
all_sub_lbls = np.array(all_sub_lbls)

tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=min(30, len(all_features)-1))
emb_2d = tsne.fit_transform(all_features)

fig, ax = plt.subplots(figsize=(9, 8))
for i, subtype in enumerate(CONFIG['subtype_order']):
    mask = (all_sub_lbls == i)
    if mask.sum() > 0:
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=subtype, alpha=0.7, s=40)
ax.set_title('t-SNE of OMNet-V3 Fused Embeddings by Subtype', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
save_figure(fig, 'tsne_embeddings.png')
plt.show()

# Section 17: Experiment Summary & Export

In [ ]:
# ============================================================
# Section 17: Experiment Summary & Export
# ============================================================

experiment_record = {
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'aggregated_results': aggregated_summary,
    'folds_completed': len(all_fold_results),
}

with open(OUTPUT_DIR / 'experiment_config.json', 'w') as f:
    json.dump(experiment_record, f, indent=2, default=str)

df_summary = pd.DataFrame(aggregated_summary).T
df_summary.to_csv(OUTPUT_DIR / 'metrics_summary.csv')

print("=" * 75)
print("  OMNet-V3 Final Execution Summary")
print("=" * 75)
print(f"  Dataset             : BreakHis ({CONFIG['dataset_id']})")
print(f"  Magnifications      : {CONFIG['magnification_levels']}")
print(f"  Folds Evaluated     : {CONFIG['n_splits']}")
print(f"  Subtype Macro-F1    : {aggregated_summary['subtype_macro_f1']['mean']:.4f} +/- {aggregated_summary['subtype_macro_f1']['std']:.4f}")
print(f"  Binary Accuracy     : {aggregated_summary['binary_accuracy']['mean']:.4f} +/- {aggregated_summary['binary_accuracy']['std']:.4f}")
print(f"  Output Directory    : {OUTPUT_DIR}")
print("=" * 75)
print("[OK] All fold checkpoints, metrics, and figures saved successfully.")